# 第 6 周可选附加——深度神经网络

只是为了从令人失望的结果中挽回一点颜面

我在 pricer/deep_neural_network.py 中训练了深度神经网络，并把权重上传到这里。请把该文件下载到 week6 目录：

文件 `deep_neural_network.pth` 在这里：

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

In [ ]:
# 导入：加载已训练的深度神经网络并做推理评估

from dotenv import load_dotenv
import os
from huggingface_hub import login
from pricer.evaluator import evaluate
from pricer.deep_neural_network import DeepNeuralNetworkRunner
from pricer.items import Item

In [ ]:
# 环境

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
# 从 Hub 加载数据划分

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
# 构建 Runner 并 setup（准备向量化与网络结构）

runner = DeepNeuralNetworkRunner(train, val[:1000])
runner.setup()

## 如果你想自己训练

那就运行这个——在我的 M1 Mac 上全力打满 GPU 大约需要 4 小时：

```python
runner.train(epochs=5)
runner.save('deep_neural_network.pth')
```

## 或者直接下载文件 `deep_neural_network.pth`：

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

并把它放到这个 week6 目录中。

In [ ]:
# 加载已保存的权重；Windows/WSL 常用带 'cpu' 的那一行
# （原英文注释说明 MAC / Win 设备差异，勿删）

#runner.load('deep_neural_network.pth')                          # 在 MAC 上加载
runner.load('deep_neural_network.pth', 'cpu')                   # use this one if want to load on a Win PC device --- if so comment out the line above

In [ ]:
# 推理封装 + 在测试集上评估深度神经网络

def deep_neural_network(item):
    return runner.inference(item)

evaluate(deep_neural_network, test)
